In [1]:
import os
import numpy as np
import pandas as pd
import segyio
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler

In [2]:
class SeismicFeatureEngineer:
    def __init__(self, input_path: str, output_path: str):
        self.input_path = input_path
        self.output_path = output_path
        self.signal_stream = []
        self.dataframe = None

    def ingest_and_preprocess(self, trace_limit: int = 2500):
        print("Reading SGY and applying standard seismic preprocessing...")
        try:
            with segyio.open(self.input_path, "r", ignore_geometry=True) as segyfile:
                segyfile.mmap()

                # 1. Extract raw traces
                raw_traces = np.array([segyfile.trace[i] for i in range(min(trace_limit, segyfile.tracecount))])

                # 2. Filter dead traces (critical for F3 dataset)
                dead_mask = np.all(raw_traces == 0, axis=1) | np.isnan(raw_traces).any(axis=1)
                valid_traces = raw_traces[~dead_mask]

                # 3. Clip amplitude spikes (prevents exploding gradients)
                perc_99 = np.percentile(np.abs(valid_traces), 99.9)
                clipped = np.clip(valid_traces, -perc_99, perc_99)

                # 4. Standardize the entire signal to Mean=0, Variance=1
                # THIS IS THE FIX: Scales the target variable properly
                flat_signal = clipped.flatten().reshape(-1, 1)
                scaler = StandardScaler()
                normalized_signal = scaler.fit_transform(flat_signal).flatten()

                self.signal_stream = normalized_signal
                print(f"Successfully processed and normalized {len(valid_traces)} valid traces.")

        except Exception as e:
            print(f"Error during ingestion: {e}")

        return self

    def engineer_features(self):
        print("Engineering time-series features from normalized signal...")
        df = pd.DataFrame({'target': self.signal_stream})

        # Simulated telemetry timestamps for the Magnora dashboard
        start_time = datetime(2026, 6, 1, 0, 0, 0)
        df['timestamp'] = [start_time + timedelta(seconds=i) for i in range(len(df))]

        # Build Lags and Rolling statistics
        df['lag_1'] = df['target'].shift(1)
        df['lag_2'] = df['target'].shift(2)
        df['lag_3'] = df['target'].shift(3)

        df['rolling_mean_5'] = df['target'].rolling(window=5).mean()
        df['rolling_std_5'] = df['target'].rolling(window=5).std()
        df['rolling_mean_10'] = df['target'].rolling(window=10).mean()

        df['signal_gradient'] = np.gradient(df['target'].fillna(0))

        self.dataframe = df.dropna().reset_index(drop=True)
        return self

    def export_data(self):
        print("Saving processed telemetry stream...")
        os.makedirs(os.path.dirname(self.output_path), exist_ok=True)

        # Reorder columns to ensure 'target' is at the end
        cols = ['timestamp'] + [col for col in self.dataframe.columns if col not in ['timestamp', 'target']] + ['target']
        self.dataframe = self.dataframe[cols]

        self.dataframe.to_csv(self.output_path, index=False)
        print(f"Data saved successfully to {self.output_path}")

In [3]:
if __name__ == "__main__":
    INPUT_FILE = '../data/f3_dataset.sgy'
    OUTPUT_FILE = '../outputs/processed_features.csv'

    pipeline = SeismicFeatureEngineer(input_path=INPUT_FILE, output_path=OUTPUT_FILE)
    pipeline.ingest_and_preprocess(trace_limit=2500).engineer_features().export_data()

Reading SGY and applying standard seismic preprocessing...
Successfully processed and normalized 2500 valid traces.
Engineering time-series features from normalized signal...
Saving processed telemetry stream...
Data saved successfully to ../outputs/processed_features.csv
